# VGG


## 0. 基本认识 VGG Overview

VGGNet 是牛津大学 Visual Geometry Group 提出的深度卷积神经网络，论文题目是 *Very Deep Convolutional Networks for Large-Scale Image Recognition*。它关注的问题是：在较统一的卷积网络设计下，继续加深网络深度是否能提高图像识别效果。

课件和论文中的核心背景可以概括为：

- VGG 参加了 2014 年 ILSVRC 图像识别竞赛，分类任务表现接近 GoogLeNet，定位任务表现很好。
- 常说的 VGG-16 指 16 个带权重的层，其中包含 13 个卷积层和 3 个全连接层。
- VGG 相比 AlexNet 更深，结构更规整，主要由重复的 VGG block 堆叠而成。
- 论文中的 VGG-16 约有 1.38 亿参数，参数量很大，但迁移到其他视觉任务时表现较好。

简单理解：VGG 的重要性不在于结构复杂，而在于它用统一的小卷积核和重复模块证明了“把网络加深”本身就很有价值。


---


## 1. 网络结构 Network Architecture

### 1.1 VGG block

VGG 的基础单元可以写成：

```text
[Conv 3x3 + ReLU] x n -> MaxPool 2x2, stride=2
```

其中：

- block 内部的卷积层一般使用 $3 \times 3$ 卷积核、stride=1、padding=1。
- padding=1 可以让卷积前后的空间尺寸保持不变。
- block 末尾使用 $2 \times 2$ 最大池化，stride=2，把宽高缩小一半。
- 通道数随着网络加深逐步增加：64 -> 128 -> 256 -> 512 -> 512。

### 1.2 VGG-16 原版结构

论文和课件中的 VGG-16 以 $3 \times 224 \times 224$ RGB 图像作为输入，输出 1000 个 ImageNet 类别。

| 顺序 | 模块 | 主要层 | 输出尺寸 |
| --- | --- | --- | --- |
| 1 | Input | RGB 图像 | $3 \times 224 \times 224$ |
| 2 | Block1 | 2 个 Conv3-64 + MaxPool | $64 \times 112 \times 112$ |
| 3 | Block2 | 2 个 Conv3-128 + MaxPool | $128 \times 56 \times 56$ |
| 4 | Block3 | 3 个 Conv3-256 + MaxPool | $256 \times 28 \times 28$ |
| 5 | Block4 | 3 个 Conv3-512 + MaxPool | $512 \times 14 \times 14$ |
| 6 | Block5 | 3 个 Conv3-512 + MaxPool | $512 \times 7 \times 7$ |
| 7 | Flatten | 展平 | $25088$ |
| 8 | FC1 | Linear + ReLU + Dropout | $4096$ |
| 9 | FC2 | Linear + ReLU + Dropout | $4096$ |
| 10 | FC3 | Linear + Softmax | $1000$ |

VGG-16 的主线结构是：

```text
2 conv -> pool
2 conv -> pool
3 conv -> pool
3 conv -> pool
3 conv -> pool
flatten -> FC -> FC -> FC
```

### 1.3 VGG 不同配置

论文中比较了多种配置，主要区别是卷积层数量不同：

| 配置 | 带权层数 | 常见名称 | 特点 |
| --- | --- | --- | --- |
| A | 11 | VGG-11 | 较浅的基准网络 |
| B | 13 | VGG-13 | 每个前两层 block 都有 2 个卷积 |
| D | 16 | VGG-16 | 最常用，13 个卷积层 + 3 个全连接层 |
| E | 19 | VGG-19 | 更深，16 个卷积层 + 3 个全连接层 |

论文结论是：在相同设计原则下，网络从 11 层加深到 16-19 层时，分类错误率会下降；但继续加深后提升会逐渐变小。


---


## 2. 尺寸和参数计算 Parameter Calculation

### 2.1 卷积输出尺寸

卷积或池化后的空间尺寸可以用下面公式计算：

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核或池化核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

VGG 中常用的 $3 \times 3$ 卷积满足 $K=3, P=1, S=1$：

$$
O = \frac{I + 2 \times 1 - 3}{1} + 1 = I
$$

所以 block 内部卷积不会改变宽高。

### 2.2 池化尺寸变化

VGG 的最大池化通常使用 $2 \times 2$，stride=2：

$$
O = \frac{I - 2}{2} + 1 = \frac{I}{2}
$$

因此输入 $224 \times 224$ 会按下面顺序缩小：

```text
224 -> 112 -> 56 -> 28 -> 14 -> 7
```

第 5 个 block 后得到 $512 \times 7 \times 7$，展平长度为：

$$
512 \times 7 \times 7 = 25088
$$

这就是原版 VGG-16 第一层全连接层输入为 25088 的原因。

### 2.3 参数量计算

卷积层参数量：

$$
K_h \times K_w \times C_{in} \times C_{out} + C_{out}
$$

其中：

- $K_h, K_w$：卷积核高和宽。
- $C_{in}$：输入通道数。
- $C_{out}$：输出通道数，也就是卷积核数量。
- 最后的 $C_{out}$ 是 bias 参数。

以配套源码第一层为例，输入通道为 1，输出通道为 64：

$$
3 \times 3 \times 1 \times 64 + 64 = 640
$$

第二个卷积层从 64 通道到 64 通道：

$$
3 \times 3 \times 64 \times 64 + 64 = 36928
$$

全连接层参数量：

$$
in\_features \times out\_features + out\_features
$$

源码把第一层全连接从原版的 4096 输出缩小到 256 输出，所以参数量明显减少：

$$
25088 \times 256 + 256 = 6422784
$$


---


## 3. 关键思想 Key Ideas

### 3.1 用小卷积核堆叠代替大卷积核

VGG 的一个重要特点是大量使用 $3 \times 3$ 小卷积核。多个 $3 \times 3$ 卷积堆叠后，可以获得更大的等效感受野：

```text
2 个 3x3 卷积 -> 等效 5x5 感受野
3 个 3x3 卷积 -> 等效 7x7 感受野
```

和直接使用大卷积核相比，小卷积核堆叠有两个好处：

- 参数更少，例如 3 个 $3 \times 3$ 卷积的参数量约为 $27C^2$，单个 $7 \times 7$ 卷积约为 $49C^2$。
- 非线性更多，因为每个卷积后面都可以接 ReLU，表达能力更强。

### 3.2 更深但更规整

VGG 的结构很统一：

- 卷积层主要都是 $3 \times 3$。
- 池化层主要都是 $2 \times 2$，stride=2。
- 每个 block 只负责提取特征，block 之间通过池化降采样。
- 最后用全连接层做分类。

这种设计让网络虽然很深，但结构容易理解和复现。后来很多模型也延续了“堆叠基础模块”的思想。

### 3.3 LRN 的作用不明显

AlexNet 中使用过 LRN Local Response Normalization。VGG 论文中比较了带 LRN 和不带 LRN 的配置，发现 LRN 没有带来明显提升，所以更深的 VGG 配置中没有继续使用 LRN。

### 3.4 深度带来的效果

论文主要验证的是深度对识别准确率的影响。结论可以记成：

- 在相同训练和评估框架下，加深网络通常能降低分类错误率。
- VGG-16 和 VGG-19 表现较好。
- 3×3 卷积贯穿全网，比只增加 1×1 卷积更能捕获空间上下文。
- 网络越深，训练成本、显存占用和参数量也越高。


---


## 4. PyTorch 实战版 PyTorch Implementation

### 4.1 原版和源码版的差别

课件和论文中的 VGG-16 是 ImageNet 版本：

```text
输入: 3 x 224 x 224
输出: 1000 类
分类器: 25088 -> 4096 -> 4096 -> 1000
```

配套源码为了在 FashionMNIST 上训练，做了简化：

```text
输入: 1 x 224 x 224
输出: 10 类
分类器: 25088 -> 256 -> 128 -> 10
```

也就是说，源码保留了 VGG-16 的 5 个卷积 block，但把输入通道和分类器改成适合 FashionMNIST 的形式。

### 4.2 模型结构

配套源码中的卷积部分可以概括为：

```python
block1: Conv2d(1, 64, 3, padding=1) -> ReLU -> Conv2d(64, 64, 3, padding=1) -> ReLU -> MaxPool2d(2, 2)
block2: Conv2d(64, 128, 3, padding=1) -> ReLU -> Conv2d(128, 128, 3, padding=1) -> ReLU -> MaxPool2d(2, 2)
block3: Conv2d(128, 256, 3, padding=1) -> ReLU -> Conv2d(256, 256, 3, padding=1) -> ReLU -> Conv2d(256, 256, 3, padding=1) -> ReLU -> MaxPool2d(2, 2)
block4: Conv2d(256, 512, 3, padding=1) -> ReLU -> Conv2d(512, 512, 3, padding=1) -> ReLU -> Conv2d(512, 512, 3, padding=1) -> ReLU -> MaxPool2d(2, 2)
block5: Conv2d(512, 512, 3, padding=1) -> ReLU -> Conv2d(512, 512, 3, padding=1) -> ReLU -> Conv2d(512, 512, 3, padding=1) -> ReLU -> MaxPool2d(2, 2)
```

分类器部分：

```python
Flatten -> Linear(7*7*512, 256) -> ReLU -> Linear(256, 128) -> ReLU -> Linear(128, 10)
```

前向传播顺序很直接：

```text
block1 -> block2 -> block3 -> block4 -> block5 -> block6
```

### 4.3 实战数据集

源码使用 `torchvision.datasets.FashionMNIST`：

- 训练集：`train=True`。
- 测试集：`train=False`。
- 图像预处理：`transforms.Resize(size=224)` 和 `transforms.ToTensor()`。
- 输入张量形状：`(B, 1, 224, 224)`。
- 输出类别数：10 类，对应 FashionMNIST 的服饰类别。

### 4.4 训练流程

训练代码的大致流程是：

1. 读取 FashionMNIST 训练集。
2. 按 8:2 划分训练集和验证集。
3. 使用 `DataLoader` 按 batch 读取数据，batch size 为 28。
4. 构建 `VGG16()` 模型，并放到 GPU 或 CPU。
5. 使用 Adam 优化器，学习率为 0.001。
6. 使用交叉熵损失 `nn.CrossEntropyLoss()`。
7. 每个 epoch 统计训练集和验证集的 loss、accuracy。
8. 保存验证集准确率最高的模型参数。
9. 用 matplotlib 画训练和验证的 loss、accuracy 曲线。

### 4.5 测试流程

测试代码的大致流程是：

1. 构建 `VGG16()` 模型。
2. 通过 `model.load_state_dict(torch.load('best_model.pth'))` 加载参数。
3. 读取 FashionMNIST 测试集。
4. 使用 `torch.no_grad()` 关闭梯度计算。
5. 前向传播得到输出 logits。
6. 使用 `torch.argmax(output, dim=1)` 得到预测类别。
7. 统计预测正确数量，计算测试准确率。
8. 随机取测试样本，打印预测类别和真实类别。

注意：模型最后一层直接输出 logits，训练时交给 `CrossEntropyLoss` 即可，不需要在模型里手动加 Softmax。


---


## 5. 注意点 Common Pitfalls

### 5.1 不要混淆原版 VGG-16 和源码版 VGG16

| 对比项 | 原版 VGG-16 | 当前源码版 |
| --- | --- | --- |
| 数据集 | ImageNet | FashionMNIST |
| 输入通道 | 3 通道 RGB | 1 通道灰度图 |
| 输入尺寸 | $3 \times 224 \times 224$ | $1 \times 224 \times 224$ |
| 输出类别 | 1000 类 | 10 类 |
| 分类器 | 4096 -> 4096 -> 1000 | 256 -> 128 -> 10 |
| Dropout | 前两个全连接层后使用 | 源码中没有使用 |

画网络结构图或计算参数时，要先确认自己采用的是哪一个版本。

### 5.2 VGG 参数量和计算量很大

VGG 的卷积结构规整，但参数量集中在后面的全连接层。原版 VGG-16 约 1.38 亿参数，训练和保存模型都比较占资源。

当前源码把全连接层改小后，参数量明显下降，但把 FashionMNIST 从 $28 \times 28$ resize 到 $224 \times 224$，计算量仍然比较大。

### 5.3 保存路径要保持一致

训练代码中保存模型的路径是：

```python
torch.save(best_model_wts, "C:/Users/86159/Desktop/VGG16/best_model.pth")
```

测试代码中加载的是：

```python
model.load_state_dict(torch.load('best_model.pth'))
```

这两个路径不一致。实际运行时更稳的做法是统一使用当前项目下的相对路径，例如 `best_model.pth`，否则测试脚本可能找不到训练得到的模型文件。

### 5.4 验证阶段可以关闭梯度

源码测试阶段已经使用 `torch.no_grad()`，这是正确的。验证阶段虽然调用了 `model.eval()`，但没有包在 `torch.no_grad()` 中。这样结果不一定错，但会多占用显存和计算图资源。

### 5.5 分类任务不要在模型末尾手动加 Softmax

使用 `nn.CrossEntropyLoss()` 时，输入应该是模型输出的原始 logits。`CrossEntropyLoss` 内部会处理 `LogSoftmax` 和负对数似然，所以模型最后一层保留 `Linear` 即可。


---


## 6. 总结 Summary

- VGGNet 的核心思想是用统一的 $3 \times 3$ 小卷积核构建更深的卷积网络。
- VGG block 通常由若干个 `Conv3x3 + ReLU` 和一个 `MaxPool2d(2, 2)` 组成。
- VGG-16 包含 13 个卷积层和 3 个全连接层，常见输入为 $3 \times 224 \times 224$，输出 1000 类。
- 5 次最大池化会把空间尺寸从 $224$ 逐步降到 $7$，最后展平为 $7 \times 7 \times 512 = 25088$。
- 小卷积核堆叠可以在减少参数的同时增加非线性，是 VGG 相比早期大卷积核网络的重要改进。
- 当前源码是 FashionMNIST 实战版，输入通道、输出类别和分类器规模都与论文原版不同。
- 阅读或复现 VGG 时，最容易错的是混淆原版结构和实战代码结构、忽略全连接层参数量、以及模型保存和加载路径不一致。


***********
